In [ ]:
import sys
sys.path.append("/exp/sbnd/data/users/lynnt/xsection/")

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import nueana as nue
from unfolding.wienersvd import WienerSVD

%load_ext autoreload
%autoreload 2

In [ ]:
# Update these paths for your local installation
checkpoint_dir = "/exp/sbnd/data/users/lynnt/xsection/notebooks/june2026/"
dfs_dir        = "/exp/sbnd/data/users/lynnt/xsection/samples/MCP2025B_v10_06_00_09/dfs_nu26/"

## load checkpoint

In [ ]:
# syst_total: dict keyed by var_save_name; SystematicsOutput for the full
# selected region (signal + background).  syst_signal and syst_bkg are
# signal- and background-only components used by the CCBC notebook.
mcmc_df, mcnue_df, syst_total, syst_signal, syst_bkg = \
    nue.load_signal_checkpoint(checkpoint_dir + "signal_checkpoint.pkl")

var_energy    = nue.electron_energy()
var_direction = nue.electron_direction()

mcbnb_pot = syst_total[var_energy.var_save_name].mcbnb_pot
# flux_norm converts raw event-rate histograms to flux-averaged units,
# which is the unit system used internally by UnfoldInput and SystematicsOutput.
flux_norm = nue.integrated_flux * (mcbnb_pot / 1e6)

## response matrices

In [ ]:
# R[i,j] = fraction of true events in true bin j reconstructed in reco bin i.
# Only signal events (signal==0) enter; background is excluded by definition.
response_energy = nue.get_response_matrix(
    reco_df = mcmc_df[mcmc_df.signal == 0],
    true_df = mcnue_df,
    var     = var_energy,
)
response_direct = nue.get_response_matrix(
    reco_df = mcmc_df[mcmc_df.signal == 0],
    true_df = mcnue_df,
    var     = var_direction,
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, var, R in [
    (axes[0], var_energy,    response_energy),
    (axes[1], var_direction, response_direct),
]:
    sns.heatmap(R.T, ax=ax, cmap='mako', annot=True, fmt='.2f',
                cbar=False, annot_kws={'fontsize': 12})
    ax.invert_yaxis()
    ax.set_xlabel(var.var_labels[1], fontsize=13)
    ax.set_ylabel(var.var_labels[2], fontsize=13)
    ax.set_xticklabels(var.bin_diff_labels, rotation=45, fontsize=9)
    ax.set_yticklabels(var.bin_diff_labels, rotation=0,  fontsize=9)
axes[0].set_title("Energy response matrix",    fontsize=14, y=1.05)
axes[1].set_title("Direction response matrix", fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

## closure test

In [ ]:
# UnfoldInput bundles the response matrix and per-key systematic covariances.
# cv_signal is stored in flux-averaged event-rate units (weights_mc / flux_norm)
# using mcbnb_pot from syst_output.  The default xsec_scale=1/NTARGETS inside
# unfold() converts both Signal, Measure, and Covariance to cross-section units.
uinp_energy = nue.UnfoldInput.build(
    var         = var_energy,
    reco_df     = mcmc_df,
    true_df     = mcnue_df,
    syst_output = syst_total[var_energy.var_save_name],
)
uinp_direct = nue.UnfoldInput.build(
    var         = var_direction,
    reco_df     = mcmc_df,
    true_df     = mcnue_df,
    syst_output = syst_total[var_direction.var_save_name],
)

In [ ]:
# Asimov closure: use the CV signal reco distribution as the "data" measurement.
# The measure must be in flux-averaged units (consistent with cv_signal).
cv_sig_df = nue.ensure_lexsorted(mcmc_df[mcmc_df.signal == 0], axis=1)

cv_meas_energy = nue.get_hist1d(
    data    = cv_sig_df[var_energy.var_evt_reco_col],
    bins    = var_energy.bins,
    weights = cv_sig_df.weights_mc / flux_norm,
)
cv_meas_direct = nue.get_hist1d(
    data    = cv_sig_df[var_direction.var_evt_reco_col],
    bins    = var_direction.bins,
    weights = cv_sig_df.weights_mc / flux_norm,
)

# MCstat only: the unfolded result should recover cv_signal exactly up to
# the smearing introduced by the WienerSVD filter (AddSmear).
result_energy = uinp_energy.unfold(WienerSVD, measure=cv_meas_energy, allowed_keys=['MCstat'])
result_direct = uinp_direct.unfold(WienerSVD, measure=cv_meas_direct, allowed_keys=['MCstat'])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
nue.plot_unfolded_result(
    result         = result_energy,
    var            = var_energy,
    truths         = {'true signal': uinp_energy.cv_signal},
    data_label     = 'unfolded Asimov (MCstat only)',
    ax             = axes[0],
    show_norm_band = False,
)
nue.plot_unfolded_result(
    result         = result_direct,
    var            = var_direction,
    truths         = {'true signal': uinp_direct.cv_signal},
    data_label     = 'unfolded Asimov (MCstat only)',
    ax             = axes[1],
    show_norm_band = False,
)
plt.tight_layout()
plt.show()

## fake data tests

In [ ]:
# FDT: scale RES (mode==1) events down by 20%.
# make_fake_data_hists reweights masked events and returns a background-subtracted
# measurement in flux-averaged units, along with the modified true histogram.
fd_meas, fd_true = nue.make_fake_data_hists(
    reco_df   = mcmc_df,
    true_df   = mcnue_df,
    var       = var_energy,
    reco_mask = mcmc_df.slc.truth.genie_mode == 1,
    true_mask = mcnue_df.genie_mode == 1,
    weight    = 0.8,
    mcbnb_pot = mcbnb_pot,
)

result = uinp_energy.unfold(WienerSVD, measure=fd_meas, allowed_keys=['MCstat', 'GENIE'])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
plt.subplots_adjust(wspace=0.3)
nue.plot_unfolded_result(
    result = result, var = var_energy,
    truths = {'baseline': uinp_energy.cv_signal, 'modified': fd_true},
    ax     = axes[0],
)
axes[0].set_title("Scale RES by \u221220% (electron energy)")
sns.heatmap(result['AddSmear'].T, ax=axes[1], cmap='mako',
            annot=True, fmt='.2f', cbar=False)
axes[1].invert_yaxis()
axes[1].set_title("Smearing matrix $A_C$")
axes[1].set_xlabel(var_energy.var_labels[1])
axes[1].set_ylabel(var_energy.var_labels[2])
axes[1].set_xticklabels(var_energy.bin_diff_labels, rotation=45, fontsize=8)
axes[1].set_yticklabels(var_energy.bin_diff_labels, rotation=0,  fontsize=8)
plt.show()

In [ ]:
# FDT: scale QE (mode==0) events up by 20% (electron direction).
fd_meas, fd_true = nue.make_fake_data_hists(
    reco_df   = mcmc_df,
    true_df   = mcnue_df,
    var       = var_direction,
    reco_mask = mcmc_df.slc.truth.genie_mode == 0,
    true_mask = mcnue_df.genie_mode == 0,
    weight    = 1.2,
    mcbnb_pot = mcbnb_pot,
)

result = uinp_direct.unfold(WienerSVD, measure=fd_meas, allowed_keys=['MCstat', 'GENIE'])
nue.plot_unfolded_result(
    result = result, var = var_direction,
    truths = {'baseline': uinp_direct.cv_signal, 'modified': fd_true},
)
plt.title("Scale QE by +20% (electron direction)")
plt.show()

In [ ]:
# FDT: scale primary pi0 production down by 50% (electron direction).
fd_meas, fd_true = nue.make_fake_data_hists(
    reco_df   = mcmc_df,
    true_df   = mcnue_df,
    var       = var_direction,
    reco_mask = mcmc_df.slc.truth.npi0 > 0,
    true_mask = mcnue_df.npi0 > 0,
    weight    = 0.5,
    mcbnb_pot = mcbnb_pot,
)

result = uinp_direct.unfold(WienerSVD, measure=fd_meas, allowed_keys=['MCstat', 'GENIE'])
nue.plot_unfolded_result(
    result = result, var = var_direction,
    truths = {'baseline': uinp_direct.cv_signal, 'modified': fd_true},
)
plt.title(r"Scale primary $\pi^0$ production by $-50\%$ (electron direction)")
plt.show()

In [ ]:
# FDT: suppress low-energy neutrinos (E_nu < 1 GeV) by 20% (electron energy).
fd_meas, fd_true = nue.make_fake_data_hists(
    reco_df   = mcmc_df,
    true_df   = mcnue_df,
    var       = var_energy,
    reco_mask = mcmc_df.slc.truth.e.genE < 1,
    true_mask = mcnue_df.e.genE < 1,
    weight    = 0.8,
    mcbnb_pot = mcbnb_pot,
)

result = uinp_energy.unfold(WienerSVD, measure=fd_meas, allowed_keys=['MCstat', 'GENIE'])
nue.plot_unfolded_result(
    result = result, var = var_energy,
    truths = {'baseline': uinp_energy.cv_signal, 'modified': fd_true},
)
plt.title(r"Suppress low-$E_\nu$ ($<1\,$GeV) by 20% (electron energy)")
plt.show()

## alternate-generator test (GiBUU)

In [ ]:
# GiBUU is a fully independent nuclear transport model used as fake data to
# probe model dependence of the unfolding (no CCBC sideband constraint here;
# see ccbc_constraint.ipynb for the constrained version).
gibuu_df, gibuu_pot, _ = nue.load_mc(
    dfs_dir + "mc_gibuu.df",
    keys=['nuecc', 'hdr', 'histpotdf'],
    cuts=nue.DEFAULT_CUTS,
)
gibuu_sig_df = nue.load_dfs(dfs_dir + "mc_gibuu.df", ['mcnuecc'], n_max_concat=np.inf)['mcnuecc']
gibuu_sig_df = nue.define_signal(gibuu_sig_df)
gibuu_sig_df = gibuu_sig_df[gibuu_sig_df.signal == 0]

gibuu_flux_norm = nue.integrated_flux * (gibuu_pot / 1e6)

In [ ]:
for var_cfg, unf_cfg, syst_bkg_out in [
    (var_energy,    uinp_energy, syst_bkg[var_energy.var_save_name]),
    (var_direction, uinp_direct, syst_bkg[var_direction.var_save_name]),
]:
    # Convert raw counts to flux-averaged units
    fd_meas_raw = nue.get_hist1d(
        data = nue.ensure_lexsorted(gibuu_df, axis=1)[var_cfg.var_evt_reco_col],
        bins = var_cfg.bins,
    )
    fd_meas  = fd_meas_raw / gibuu_flux_norm
    fd_meas -= syst_bkg_out.rate_hist_cv  # subtract CV background

    fd_true = nue.get_hist1d(
        data = nue.ensure_lexsorted(gibuu_sig_df, axis=1)[var_cfg.var_nu_col],
        bins = var_cfg.bins,
    ) / gibuu_flux_norm

    # Poisson stat covariance on the GiBUU signal-region observation
    add_cov = np.diag(fd_meas_raw) / gibuu_flux_norm ** 2

    result = unf_cfg.unfold(
        WienerSVD,
        measure      = fd_meas,
        allowed_keys = ['MCstat', 'GENIE'],
        extra_cov    = add_cov,
    )
    nue.plot_unfolded_result(
        result     = result,
        var        = var_cfg,
        truths     = {'baseline (GENIE)': unf_cfg.cv_signal, 'GiBUU truth': fd_true},
        data_label = 'unfolded GiBUU',
    )
    plt.title(f"GiBUU alternate-generator test ({var_cfg.var_plot_name})")
    plt.show()